In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn #neural network module
import torch.optim as optim #optimizer module
import torchvision.models as models#contains all models 
import torchvision.transforms as transforms#daata augmentation and normalising numbers
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader#loads imgages into batches of 32 each time and shuffles
from sklearn.metrics import classification_report , confusion_matrix#reports precision and recall and confusion matrix
from torchvision.transforms import v2
from torchvision import datasets
print('done')

done


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")#uses GPU for computation if available
print(device)

cuda


In [4]:
train_transforms = transforms.Compose([v2.Resize((224,224)),#resize images to 224x224 pixels
                    v2.ToImage(),
                    v2.ToDtype(torch.float32, scale=True),#scales values from 0 to 1
                    v2.Normalize(mean=[0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225])])#normalise values for resnet

train_ds = datasets.ImageFolder(root='/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/train',transform=train_transforms,
is_valid_file=lambda path: '__MACOSX' not in path and not os.path.basename(path).startswith('.'))#(correct file path),loading images for training
train_loader = DataLoader(train_ds,32,shuffle=True,num_workers=2,pin_memory=True)#load minibatches of test data and store in train_loader
print(len(train_loader))

test_ds = datasets.ImageFolder(root='/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/test',transform=train_transforms,
                     is_valid_file=lambda path: '__MACOSX' not in path and not os.path.basename(path).startswith('.'))#loading images for testing
test_loader = DataLoader(test_ds,32,shuffle=False,num_workers=2,pin_memory=True)#load minibatches of test data and store in test_loader
print(len(test_loader))

163
20


In [6]:
total = np.bincount(train_ds.targets)
normal_total = total[0]
pneumonia_total = total[1]
normal_weight = (normal_total + pneumonia_total) / (2*normal_total)
pneumonia_weight = (normal_total + pneumonia_total) / (2*pneumonia_total)
print(normal_weight)
print(pneumonia_weight)

1.9448173005219984
0.6730322580645162


In [7]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)#resnet model
for parameter in model.parameters():
    parameter.requires_grad = False#Freezes brackprop for fully connected layers(False),setting to true to learn features better
new_head = nn.Linear(in_features=512,out_features=2)
model.fc = new_head
model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s] 


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [20]:
class_weights = torch.tensor([normal_weight,pneumonia_weight],dtype=torch.float32).to(device)
loss_criterion = nn.CrossEntropyLoss(weight=class_weights)#loss funciton
optimizer = optim.Adam(model.fc.parameters(),lr=0.001)#SGD
print('done')


done


In [ ]:
for parameter in model.parameters():
    parameter.requires_grad = True
optimizer = optim.Adam(model.parameters(),lr=1e-5)


In [21]:
model.train()
total_epochs = 10
for epoch in range(total_epochs):
    epoch_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device='cuda')
        labels = labels.to(device='cuda')
        optimizer.zero_grad()#reset and clear previous gradients
        predicted_logits = model(images)
        loss = loss_criterion(predicted_logits,labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    average_loss = epoch_loss / len(train_loader)
    print("Epoch number", epoch, ": average loss-",average_loss)
    

Epoch number 0 : average loss- 0.26753768914499165
Epoch number 1 : average loss- 0.18877534328337095
Epoch number 2 : average loss- 0.16200480658126756
Epoch number 3 : average loss- 0.14993888553391937
Epoch number 4 : average loss- 0.14774898377938506
Epoch number 5 : average loss- 0.13171359253304502
Epoch number 6 : average loss- 0.12250037326768863
Epoch number 7 : average loss- 0.12480388425199167
Epoch number 8 : average loss- 0.11965312715619802
Epoch number 9 : average loss- 0.11594909979201899


In [22]:
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for images,labels in test_loader:
        images = images.to(device='cuda')
        labels = labels.to(device='cuda')
        outputs = model(images)
        preds = torch.argmax(outputs , dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
confusion_matrix(all_labels,all_preds)
classes = ['normal','pneumonia']
print(classification_report(all_labels,all_preds,target_names=classes))

              precision    recall  f1-score   support

      normal       0.94      0.58      0.72       234
   pneumonia       0.80      0.98      0.88       390

    accuracy                           0.83       624
   macro avg       0.87      0.78      0.80       624
weighted avg       0.85      0.83      0.82       624

